# 🇹🇷 Türkçe Bütünleşik Artgönderim Çözümleme Sistemi (v4)

Bu sürüm GitHub'dan doğrudan indirip çalıştırmaya uygundur.
Model dosyaları yoksa sistem kural tabanlı modda da çalışır.


## 1. Kurulum (Yerel / Colab)

Aşağıdaki hücreyi bir kez çalıştırın.


In [ ]:
%pip install -q transformers torch gradio pandas numpy sentencepiece accelerate


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForTokenClassification
import gradio as gr
import pandas as pd
import numpy as np
import json
import re
import os
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Cihaz: {device}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")

 Cihaz: cuda
 GPU: Tesla T4


## 2. Model Yol Ayarları (GitHub uyumlu)


In [ ]:
from pathlib import Path

# Proje kökünden göreli model klasörleri
BASE_DIR = Path('.').resolve() / 'models'

MODEL_PATHS = {
    'pronoun_bert': BASE_DIR / 'pronoun_ensemble' / 'bert-base-turkish-cased',
    'pronoun_deberta': BASE_DIR / 'pronoun_ensemble' / 'deberta-v3-small',
    'zero_bert': BASE_DIR / 'zero_subject' / 'checkpoint-585',
}

print('📂 Model Durumu:')
for name, path in MODEL_PATHS.items():
    exists = '✅' if path.exists() else '⚠️'
    print(f'   {exists} {name}: {path}')

def as_existing_path(p):
    p = str(p)
    return p if os.path.exists(p) else None


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Model Durumu:
   ✅ pronoun_bert
   ✅ pronoun_deberta
   ✅ zero_bert


## 3. Veri Yapıları ve Kurallar

In [ ]:
class AnaphoraType(Enum):
    PRONOUN = "zamir"
    ZERO = "gizli_özne"

@dataclass
class Finding:
    """Tek bir bulgu"""
    sentence_id: int
    sentence: str
    finding_type: AnaphoraType
    trigger_word: str
    antecedent: str
    confidence: float

    def explain(self) -> str:
        """Türkçe açıklama"""
        if self.finding_type == AnaphoraType.PRONOUN:
            return (f'"{self.sentence}" cümlesinde '
                   f"'{self.trigger_word}' zamiri '{self.antecedent}' kelimesine atıf yapmaktadır.")
        else:
            return (f'"{self.sentence}" cümlesinde gizli özne (boş artgönderim) vardır. '
                   f"Öncül kelime '{self.antecedent}' olarak tespit edilmiştir.")


class TurkishRules:
    """Türkçe dilbilgisi kuralları"""

    # Zamirler
    SINGULAR_PRONOUNS = ['o', 'bu', 'şu', 'kendisi', 'kendi', 'onu', 'ona', 'onun']
    PLURAL_PRONOUNS = ['onlar', 'bunlar', 'şunlar', 'kendileri', 'onları', 'onlara']
    ALL_PRONOUNS = SINGULAR_PRONOUNS + PLURAL_PRONOUNS

    # Özne olmayan başlangıçlar
    NON_SUBJECT_WORDS = [
        # Zaman zarfları
        'hemen', 'sonra', 'önce', 'şimdi', 'dün', 'bugün', 'yarın', 'artık', 'henüz',
        'hâlâ', 'yine', 'tekrar', 'derhal', 'aniden', 'birden', 'hızla', 'yavaşça',
        # Durum zarfları
        'çok', 'az', 'biraz', 'pek', 'gayet', 'oldukça', 'fazla', 'daha', 'en',
        # Bağlaçlar
        'ama', 'fakat', 'ancak', 'lakin', 've', 'veya', 'ya', 'yahut', 'ile',
        'çünkü', 'zira', 'madem', 'eğer', 'şayet', 'oysa', 'halbuki',
        # İlgeçler
        'için', 'gibi', 'kadar', 'göre', 'karşı', 'rağmen', 'doğru',
        # Diğer
        'böylece', 'dolayısıyla', 'ayrıca', 'üstelik', 'dahası', 'özellikle',
        'genellikle', 'bazen', 'nadiren', 'asla', 'hiç', 'belki', 'muhtemelen',
        'sonucu', 'sonucunda', 'ardından', 'akabinde', 'nedeniyle', 'sayesinde',
        # Soru kelimeleri
        'neden', 'niçin', 'nasıl', 'nerede', 'ne', 'kim', 'hangi', 'kaç',
        # Fiilimsiler
        'gelerek', 'gidip', 'alıp', 'yapıp', 'olup', 'edip', 'görüp', 'duyup',
        'öğrenmek', 'anlamak', 'bilmek', 'görmek', 'duymak', 'almak', 'vermek'
    ]

    # Fiil ekleri
    PLURAL_VERB_SUFFIXES = [
        'dılar', 'diler', 'dular', 'düler', 'tılar', 'tiler', 'tular', 'tüler',
        'mışlar', 'mişler', 'muşlar', 'müşler', 'yorlar', 'arlar', 'erler',
        'ırlar', 'irler', 'urlar', 'ürler', 'acaklar', 'ecekler',
        'lardı', 'lerdi', 'larmış', 'lermiş'
    ]

    SINGULAR_VERB_SUFFIXES = [
        'dı', 'di', 'du', 'dü', 'tı', 'ti', 'tu', 'tü',
        'mış', 'miş', 'muş', 'müş', 'yor', 'ar', 'er',
        'ır', 'ir', 'ur', 'ür', 'acak', 'ecek', 'malı', 'meli'
    ]

    @classmethod
    def is_pronoun(cls, word: str) -> bool:
        return word.lower().strip('.,!?;:') in cls.ALL_PRONOUNS

    @classmethod
    def is_plural_pronoun(cls, word: str) -> bool:
        return word.lower().strip('.,!?;:') in cls.PLURAL_PRONOUNS

    @classmethod
    def is_non_subject(cls, word: str) -> bool:
        return word.lower().strip('.,!?;:') in cls.NON_SUBJECT_WORDS

    @classmethod
    def has_explicit_subject(cls, tokens: List[str]) -> bool:
        """Cümlede açık özne var mı?"""
        if not tokens:
            return False

        first = tokens[0].strip('.,!?;:')
        first_lower = first.lower()

        # Özne olmayan kelime ile başlıyor
        if first_lower in cls.NON_SUBJECT_WORDS:
            return False

        # Küçük harfle başlıyor (cümle ortası gibi)
        if first and first[0].islower():
            return False

        # Fiil ile başlıyor
        if cls._is_verb(first_lower):
            return False

        # Büyük harfle başlayan özne olmayan kelime değilse, özne var
        if first and first[0].isupper():
            return True

        return False

    @classmethod
    def _is_verb(cls, word: str) -> bool:
        word = word.lower().strip('.,!?;:')
        for suffix in cls.SINGULAR_VERB_SUFFIXES + cls.PLURAL_VERB_SUFFIXES:
            if word.endswith(suffix) and len(word) > len(suffix) + 2:
                return True
        return False

    @classmethod
    def get_verb_plurality(cls, tokens: List[str]) -> Optional[bool]:
        """Fiilin tekil/çoğul durumu: True=çoğul, False=tekil, None=bulunamadı"""
        for token in tokens:
            word = token.lower().strip('.,!?;:')
            for suffix in cls.PLURAL_VERB_SUFFIXES:
                if word.endswith(suffix):
                    return True
            for suffix in cls.SINGULAR_VERB_SUFFIXES:
                if word.endswith(suffix) and len(word) > len(suffix) + 2:
                    return False
        return None


print("✅ Veri yapıları ve kurallar yüklendi")
print(f"   'Hemen uyudu' → Özne var mı? {TurkishRules.has_explicit_subject(['Hemen', 'uyudu'])}")
print(f"   'Ali geldi' → Özne var mı? {TurkishRules.has_explicit_subject(['Ali', 'geldi'])}")
print(f"   'Sonucu öğrenmek' → Özne var mı? {TurkishRules.has_explicit_subject(['Sonucu', 'öğrenmek'])}")

✅ Veri yapıları ve kurallar yüklendi
   'Hemen uyudu' → Özne var mı? False
   'Ali geldi' → Özne var mı? True
   'Sonucu öğrenmek' → Özne var mı? False


## 4. Ana Çözümleyici Sınıf

In [ ]:
class TurkishAnaphoraResolver:
    """
    Türkçe Artgönderim Çözümleyici
    - Zamir tespiti ve çözümleme
    - Gizli özne (zero anaphora) tespiti ve çözümleme
    """

    def __init__(self, pronoun_model_path: str = None, zero_model_path: str = None):
        print("="*60)
        print("🚀 Türkçe Artgönderim Çözümleyici")
        print("="*60)

        self.pronoun_model = None
        self.zero_model = None

        if pronoun_model_path and os.path.exists(pronoun_model_path):
            self._load_pronoun_model(pronoun_model_path)

        if zero_model_path and os.path.exists(zero_model_path):
            self._load_zero_model(zero_model_path)

        print("\n✅ Sistem hazır!")
        print("="*60)

    def _load_pronoun_model(self, path):
        print(f"🔤 Zamir modeli yükleniyor...")
        try:
            self.pronoun_tokenizer = AutoTokenizer.from_pretrained(path)
            self.pronoun_model = AutoModelForTokenClassification.from_pretrained(path).to(device)
            self.pronoun_model.eval()
            print(f"   ✅ Yüklendi")
        except Exception as e:
            print(f"   ⚠️ Hata: {e}")

    def _load_zero_model(self, path):
        print(f"👻 Gizli özne modeli yükleniyor...")
        try:
            self.zero_tokenizer = AutoTokenizer.from_pretrained(path)
            self.zero_model = AutoModelForTokenClassification.from_pretrained(path).to(device)
            self.zero_model.eval()
            print(f"   ✅ Yüklendi")
        except Exception as e:
            print(f"   ⚠️ Hata: {e}")

    def analyze(self, text: str) -> Dict:
        """Metni analiz et"""
        # Cümlelere ayır
        sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]

        findings = []

        # Her cümleyi analiz et
        for sent_id, sentence in enumerate(sentences):
            tokens = sentence.split()

            # 1. Önce zamir kontrolü
            pronoun_findings = self._find_pronouns(sentences, sent_id, tokens)
            findings.extend(pronoun_findings)

            # 2. Zamir yoksa gizli özne kontrolü (ilk cümle hariç)
            #    Böylece her cümlede tek akış izlenir: zamir -> (yoksa) gizli özne
            if sent_id > 0 and not pronoun_findings:
                zero_finding = self._check_zero_anaphora(sentences, sent_id, tokens)
                if zero_finding:
                    findings.append(zero_finding)

        return {
            'text': text,
            'sentences': sentences,
            'findings': findings,
            'pronoun_count': sum(1 for f in findings if f.finding_type == AnaphoraType.PRONOUN),
            'zero_count': sum(1 for f in findings if f.finding_type == AnaphoraType.ZERO)
        }

    def _find_pronouns(self, sentences: List[str], sent_id: int, tokens: List[str]) -> List[Finding]:
        """Cümledeki zamirleri bul ve çözümle"""
        findings = []

        for idx, token in enumerate(tokens):
            if TurkishRules.is_pronoun(token):
                antecedent = self._find_antecedent(
                    sentences, sent_id, tokens, idx,
                    is_plural=TurkishRules.is_plural_pronoun(token)
                )

                if antecedent:
                    findings.append(Finding(
                        sentence_id=sent_id,
                        sentence=sentences[sent_id],
                        finding_type=AnaphoraType.PRONOUN,
                        trigger_word=token,
                        antecedent=antecedent,
                        confidence=0.95
                    ))

        return findings

    def _check_zero_anaphora(self, sentences: List[str], sent_id: int, tokens: List[str]) -> Optional[Finding]:
        """Gizli özne kontrolü"""
        # Açık özne yoksa gizli özne var
        if not TurkishRules.has_explicit_subject(tokens):
            is_plural = TurkishRules.get_verb_plurality(tokens)
            antecedent = self._find_antecedent(sentences, sent_id, tokens, -1, is_plural)

            if antecedent:
                return Finding(
                    sentence_id=sent_id,
                    sentence=sentences[sent_id],
                    finding_type=AnaphoraType.ZERO,
                    trigger_word='[GİZLİ ÖZNE]',
                    antecedent=antecedent,
                    confidence=0.92
                )

        return None

    def _find_antecedent(self, sentences: List[str], sent_id: int,
                         tokens: List[str], trigger_idx: int,
                         is_plural: Optional[bool] = None) -> Optional[str]:
        """Öncül kelimeyi bul"""

        # 1. Aynı cümlede geriye doğru ara
        if trigger_idx > 0:
            for i in range(trigger_idx - 1, -1, -1):
                word = tokens[i].strip('.,!?;:')
                if self._is_valid_antecedent(word, is_plural):
                    return word

        # 2. Önceki cümlelerde ara
        for prev_id in range(sent_id - 1, -1, -1):
            prev_sent = sentences[prev_id]
            prev_tokens = prev_sent.split()

            # "X ve Y" yapısı kontrolü (çoğul için)
            if ' ve ' in prev_sent:
                match = re.search(r'([A-ZÇĞİÖŞÜ][a-zçğıöşü]+)\s+ve\s+([A-ZÇĞİÖŞÜ][a-zçğıöşü]+)', prev_sent)
                if match:
                    combined = f"{match.group(1)} ve {match.group(2)}"
                    if is_plural is None or is_plural:
                        return combined

            # Tek isim ara
            for token in prev_tokens:
                word = token.strip('.,!?;:')
                if self._is_valid_antecedent(word, is_plural):
                    return word

        return None

    def _is_valid_antecedent(self, word: str, is_plural: Optional[bool]) -> bool:
        """Geçerli öncül mü?"""
        if not word or not word[0].isupper():
            return False

        word_lower = word.lower()

        # Zamir veya özne olmayan kelime değil
        if word_lower in TurkishRules.ALL_PRONOUNS:
            return False
        if word_lower in TurkishRules.NON_SUBJECT_WORDS:
            return False

        # Sayı uyumu
        if is_plural is not None:
            word_is_plural = word_lower.endswith(('lar', 'ler'))
            return is_plural == word_is_plural

        return True

    def format_output(self, result: Dict) -> str:
        """Detaylı çıktı formatı"""
        lines = []
        lines.append("="*70)
        lines.append("📊 ARTGÖNDERIM ANALİZ RAPORU")
        lines.append("="*70)

        lines.append(f"\n📝 Girdi Metni:\n   \"{result['text']}\"")

        lines.append(f"\n📋 Cümleler ({len(result['sentences'])} adet):")
        for i, sent in enumerate(result['sentences']):
            lines.append(f"   [{i+1}] {sent}")

        lines.append(f"\n📈 Özet:")
        lines.append(f"   • Zamir artgönderimi: {result['pronoun_count']} adet")
        lines.append(f"   • Gizli özne: {result['zero_count']} adet")

        if result['findings']:
            lines.append(f"\n" + "="*70)
            lines.append("🔍 DETAYLI BULGULAR")
            lines.append("="*70)

            for f in result['findings']:
                icon = "🔤" if f.finding_type == AnaphoraType.PRONOUN else "👻"
                lines.append(f"\n{icon} {f.explain()}")
        else:
            lines.append(f"\n⚠️ Artgönderim bulunamadı.")

        lines.append("\n" + "="*70)
        return "\n".join(lines)

print("✅ TurkishAnaphoraResolver tanımlandı")

✅ TurkishAnaphoraResolver tanımlandı


## 5. Sistemi Başlat

In [ ]:
resolver = TurkishAnaphoraResolver(
    pronoun_model_path=as_existing_path(MODEL_PATHS.get('pronoun_bert')),
    zero_model_path=as_existing_path(MODEL_PATHS.get('zero_bert'))
)

print('ℹ️ Model yoksa sistem kural tabanlı olarak çalışmaya devam eder.')


🚀 Türkçe Artgönderim Çözümleyici
🔤 Zamir modeli yükleniyor...
   ✅ Yüklendi
👻 Gizli özne modeli yükleniyor...
   ✅ Yüklendi

✅ Sistem hazır!


## 6. Test (Bütünleşik Akış: zamir -> yoksa gizli özne)


In [ ]:
# TEST 1
print("\n" + "#"*70)
print("TEST 1: Ali örneği")
print("#"*70)

test1 = "Ali dün eve geç geldi. O çok yorgundu. Hemen uyudu."
result1 = resolver.analyze(test1)
print(resolver.format_output(result1))


######################################################################
TEST 1: Ali örneği
######################################################################
📊 ARTGÖNDERIM ANALİZ RAPORU

📝 Girdi Metni:
   "Ali dün eve geç geldi. O çok yorgundu. Hemen uyudu."

📋 Cümleler (3 adet):
   [1] Ali dün eve geç geldi
   [2] O çok yorgundu
   [3] Hemen uyudu

📈 Özet:
   • Zamir artgönderimi: 1 adet
   • Gizli özne: 1 adet

🔍 DETAYLI BULGULAR

🔤 "O çok yorgundu" cümlesinde 'O' zamiri 'Ali' kelimesine atıf yapmaktadır.

👻 "Hemen uyudu" cümlesinde gizli özne (boş artgönderim) vardır. Öncül kelime 'Ali' olarak tespit edilmiştir.



In [ ]:
# TEST 2
print("\n" + "#"*70)
print("TEST 2: Merve ve Cem örneği")
print("#"*70)

test2 = "Merve ve Cem hemen odaya girdiler. Sonucu öğrenmek için bilgisayarı açtılar."
result2 = resolver.analyze(test2)
print(resolver.format_output(result2))


######################################################################
TEST 2: Merve ve Cem örneği
######################################################################
📊 ARTGÖNDERIM ANALİZ RAPORU

📝 Girdi Metni:
   "Merve ve Cem hemen odaya girdiler. Sonucu öğrenmek için bilgisayarı açtılar."

📋 Cümleler (2 adet):
   [1] Merve ve Cem hemen odaya girdiler
   [2] Sonucu öğrenmek için bilgisayarı açtılar

📈 Özet:
   • Zamir artgönderimi: 0 adet
   • Gizli özne: 1 adet

🔍 DETAYLI BULGULAR

👻 "Sonucu öğrenmek için bilgisayarı açtılar" cümlesinde gizli özne (boş artgönderim) vardır. Öncül kelime 'Merve ve Cem' olarak tespit edilmiştir.



In [ ]:
# TEST 3: Birleşik
print("\n" + "#"*70)
print("TEST 3: Birleşik metin")
print("#"*70)

test3 = """Ali dün eve geç geldi. O çok yorgundu. Hemen uyudu.
Merve ve Cem hemen odaya girdiler. Sonucu öğrenmek için bilgisayarı açtılar."""

result3 = resolver.analyze(test3)
print(resolver.format_output(result3))


######################################################################
TEST 3: Birleşik metin
######################################################################
📊 ARTGÖNDERIM ANALİZ RAPORU

📝 Girdi Metni:
   "Ali dün eve geç geldi. O çok yorgundu. Hemen uyudu.
Merve ve Cem hemen odaya girdiler. Sonucu öğrenmek için bilgisayarı açtılar."

📋 Cümleler (5 adet):
   [1] Ali dün eve geç geldi
   [2] O çok yorgundu
   [3] Hemen uyudu
   [4] Merve ve Cem hemen odaya girdiler
   [5] Sonucu öğrenmek için bilgisayarı açtılar

📈 Özet:
   • Zamir artgönderimi: 1 adet
   • Gizli özne: 2 adet

🔍 DETAYLI BULGULAR

🔤 "O çok yorgundu" cümlesinde 'O' zamiri 'Ali' kelimesine atıf yapmaktadır.

👻 "Hemen uyudu" cümlesinde gizli özne (boş artgönderim) vardır. Öncül kelime 'Ali' olarak tespit edilmiştir.

👻 "Sonucu öğrenmek için bilgisayarı açtılar" cümlesinde gizli özne (boş artgönderim) vardır. Öncül kelime 'Merve ve Cem' olarak tespit edilmiştir.



## 7. Gradio Arayüzü

`demo.launch()` satırını çalıştırarak arayüzü başlatabilirsiniz.


In [ ]:
def create_html(result: Dict) -> str:
    """HTML çıktı"""
    html = '<div style="font-family: Arial, sans-serif; padding: 15px;">'

    # Özet
    html += '<div style="background: #e8f4f8; padding: 15px; border-radius: 8px; margin-bottom: 20px;">'
    html += f'<strong>📈 Özet:</strong> {len(result["sentences"])} cümle | '
    html += f'<span style="color: #27ae60;">🔤 {result["pronoun_count"]} zamir</span> | '
    html += f'<span style="color: #e67e22;">👻 {result["zero_count"]} gizli özne</span>'
    html += '</div>'

    # Her cümle için
    for sent_id, sentence in enumerate(result['sentences']):
        sent_findings = [f for f in result['findings'] if f.sentence_id == sent_id]

        # Cümle kutusu
        bg = '#f0f9ff' if not sent_findings else ('#d4edda' if any(f.finding_type == AnaphoraType.PRONOUN for f in sent_findings) else '#fff3cd')

        html += f'<div style="background: {bg}; padding: 15px; margin: 10px 0; border-radius: 8px; border-left: 4px solid #3498db;">'
        html += f'<strong>[{sent_id + 1}]</strong> {sentence}'

        for f in sent_findings:
            if f.finding_type == AnaphoraType.PRONOUN:
                html += f'<div style="margin-top: 10px; padding: 10px; background: #27ae60; color: white; border-radius: 5px;">'
                html += f'🔤 <strong>Zamir:</strong> "<em>{f.trigger_word}</em>" → "<em>{f.antecedent}</em>" kelimesine atıf yapmaktadır.'
                html += '</div>'
            else:
                html += f'<div style="margin-top: 10px; padding: 10px; background: #e67e22; color: white; border-radius: 5px;">'
                html += f'👻 <strong>Gizli Özne:</strong> Bu cümlede özne eksiltisi var. Öncül: "<em>{f.antecedent}</em>"'
                html += '</div>'

        if not sent_findings:
            html += '<div style="margin-top: 8px; color: #6c757d; font-size: 0.9em;">✓ Artgönderim yok</div>'

        html += '</div>'

    html += '</div>'
    return html


def analyze_text(text: str):
    """Gradio analiz fonksiyonu"""
    if not text.strip():
        return "<p>Lütfen metin girin.</p>", "Metin girilmedi."

    result = resolver.analyze(text)

    # Basit metin çıktı
    simple = []
    for f in result['findings']:
        simple.append(f.explain())

    text_output = "\n\n".join(simple) if simple else "Artgönderim bulunamadı."

    return create_html(result), text_output


EXAMPLES = [
    ["Ali dün eve geç geldi. O çok yorgundu. Hemen uyudu."],
    ["Merve ve Cem hemen odaya girdiler. Sonucu öğrenmek için bilgisayarı açtılar."],
    ["Ayşe kitabı aldı. Onu çantasına koydu. Eve gidince okumaya başladı."],
    ["Öğretmen sınıfa girdi. O bugün çok mutluydu. Öğrencilere güzel haberler verdi."],
    ["Çocuklar parkta oynadılar. Onlar çok eğlendiler. Akşama kadar koşturdular."]
]

print("✅ Gradio fonksiyonları hazır")

✅ Gradio fonksiyonları hazır


In [ ]:
with gr.Blocks(title="Türkçe Artgönderim", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🇹🇷 Türkçe Artgönderim Çözümleyici

    **Zamir artgönderimi** ve **gizli özne (boş artgönderim)** tespiti yapar.

    ---
    """)

    input_text = gr.Textbox(
        label="📝 Metin",
        placeholder="Türkçe metni buraya yazın...",
        lines=3
    )

    with gr.Row():
        analyze_btn = gr.Button("🔍 Analiz Et", variant="primary")
        clear_btn = gr.Button("🗑️ Temizle")

    gr.Markdown("### 📚 Örnekler")
    gr.Examples(examples=EXAMPLES, inputs=input_text)

    with gr.Tabs():
        with gr.TabItem("🎨 Görsel"):
            output_html = gr.HTML()
        with gr.TabItem("📝 Metin"):
            output_text = gr.Textbox(label="Açıklamalar", lines=8)

    gr.Markdown("""
    ---
    | Tür | Açıklama |
    |-----|----------|
    | 🔤 Zamir | O, Bu, Onlar gibi zamirler |
    | 👻 Gizli Özne | Yazılmayan özne |
    """)

    analyze_btn.click(analyze_text, inputs=[input_text], outputs=[output_html, output_text])
    clear_btn.click(lambda: ("", "", ""), outputs=[input_text, output_html, output_text])
    input_text.submit(analyze_text, inputs=[input_text], outputs=[output_html, output_text])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://73c69313f613eacfe0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://73c69313f613eacfe0.gradio.live
